# Stage 1.4.2.5.2 — Docling for Scanned PDFs

For Stage 1.4.2.5.2 — Docling for Scanned PDFs, we'll use your existing scanned PDF directly. We will not manually convert it to scanned_page.png first. Docling's DocumentConverter can take a PDF directly, and its PDF pipeline supports OCR for scanned/image-based content.

        Stage 1.4.2 — OCR & Complex PDF Ingestion
        │
        ├── Stage 1.4.2.1
        │   Text-based PDF vs Scanned PDF
        │
        ├── Stage 1.4.2.2
        │   Detecting Scanned / Non-Text PDF Pages
        │
        ├── Stage 1.4.2.3
        │   OCR Fundamentals
        │
        ├── Stage 1.4.2.4
        │   OCR Engines & Document Processing Libraries
        │
        └── Stage 1.4.2.5
            Modern OCR / Document Understanding Approach
            │
            ├── Stage 1.4.2.5.1
            │   Basic OCR with Tesseract — Conceptual + Minimal Demo
            │
            ├── Stage 1.4.2.5.2
            │   Docling for Scanned PDFs
            │
            ├── Stage 1.4.2.5.3
            │   Docling Layout & Reading Order
            │
            ├── Stage 1.4.2.5.4
            │   Docling Table / Image / Formula Understanding
            │
            └── Stage 1.4.2.5.5
                Docling → LangChain → RAG


Goal

Our experiment will be:

    Scanned PDF
        │
        ▼
    Docling
        │
        ├── PDF processing
        ├── OCR
        ├── Layout understanding
        └── Document representation
        │
        ▼
    DoclingDocument
        │
        ├── Markdown
        ├── JSON
        └── Text

This is much closer to how we want to approach document ingestion in a production RAG system.

### Step 1 — Install Docling

From the root of your existing rag-learning project, run:

uv add docling

This is the installation command recommended by the current Docling documentation, and Docling supports Windows.

After installation, verify:

uv run python -c "import docling; print('Docling installed successfully')"

You should get:

Docling installed successfully
Important

At this stage, don't install pytesseract just for this exercise unless you already installed it from the previous lesson.

Docling itself supports multiple OCR engines, including Tesseract, RapidOCR and EasyOCR.

For our first Docling experiment, we'll start with Docling's standard PDF pipeline rather than manually wiring an OCR engine.

### Step 2 — Verify the Notebook Kernel

Open your existing notebook in VS Code.

Make sure the selected kernel is the same Python environment managed by your uv project.

Run:

In [ ]:
import sys

print(sys.executable)

This is important because we want:

VS Code Notebook
       ↓
your UV environment
       ↓
Docling

and not some unrelated global Python installation.

### Step 3 — Import DocumentConverter

Now create a new notebook cell:

In [ ]:
from docling.document_converter import DocumentConverter

### Step 4 — Define the Scanned PDF

We are going to use the scanned PDF you already have.

Don't create an image manually.

For example:

In [ ]:
from pathlib import Path

pdf_path = Path("D:\\AI Learning\\rag-learning\\data\\raw\\pdf\\azure_event_hubs_scanned.pdf")

print(pdf_path.exists())
print(pdf_path)

We want:

True

If you don't remember the exact location, we can identify it from your existing notebook/project files rather than guessing the path.

### Step 5 — Create the Docling Converter

Now:

In [ ]:
converter = DocumentConverter()

This is the central object we'll use.

Conceptually:

       DocumentConverter
              │
              ├── PDF
              ├── DOCX
              ├── PPTX
              ├── XLSX
              ├── Images
              └── other supported formats

Docling's DocumentConverter is its primary entry point for converting documents into its unified document representation.

### Step 6 — Convert the Scanned PDF

Now the important line:

In [ ]:
result = converter.convert(pdf_path)

That's it.

Notice what we didn't write:

     ❌ PDF → PNG manually
     ❌ Pillow
     ❌ pytesseract.image_to_string()
     ❌ manually combine pages

Instead:

     Scanned PDF
          ↓
     DocumentConverter
          ↓
     Docling processing pipeline

### Step 7 — Understand What result Is

Let's inspect:

In [ ]:
print(type(result))

You'll get a Docling conversion result object.

Now:

In [ ]:
print(result.status)

And:

In [ ]:
print(result.document)

The important object is:

In [ ]:
result.document

This is a DoclingDocument.

Conceptually:

    result
    │
    ├── status
    ├── input information
    └── document
        │
        └── DoclingDocument

### Step 8 — Export the Result as Markdown

Now let's see what Docling understood.

In [ ]:
markdown_text = result.document.export_to_markdown()

Display it:

In [ ]:
print(markdown_text)

This is where things become interesting.

Instead of getting only raw OCR text, we're asking Docling to represent the processed document in Markdown.

Docling's standard usage explicitly supports converting a document and exporting the resulting DoclingDocument to Markdown

### Step 9 — Save the Markdown

Let's preserve the result:

In [ ]:
markdown_path = Path("scanned_document_docling.md")

markdown_path.write_text(
    markdown_text,
    encoding="utf-8"
)

print(f"Saved: {markdown_path}")

Now we have:

    scanned.pdf
        │
        ▼
        Docling
        │
        ▼
    scanned_document_docling.md

### Step 10 — Export as JSON

Now let's look at another important representation.

In [ ]:
import json

# Export to a Python dictionary, then serialize to JSON text
json_text = json.dumps(result.document.export_to_dict(), indent=2)
print( json_text )

Save it

In [ ]:
json_path = Path("scanned_document_docling.json")

json_path.write_text(
    json_text,
    encoding="utf-8"
)

print(f"Saved: {json_path}")

Now:

      Scanned PDF
         │
         ▼     
      DoclingDocument
         │
         ├── Markdown
         │
         └── JSON

Docling's newer document model is specifically designed as a universal representation that can preserve document hierarchy and can be exported to formats including JSON and Markdown.

### Step 11 — Inspect the JSON

Let's see what we have.

In [ ]:
import json

doc_json = json.loads(json_text)

print(type(doc_json))

In [ ]:
print(doc_json.keys())

Don't worry if the exact keys differ from what you expect.

The important conceptual difference is:

Basic OCR

    Image
      ↓
    Text
Docling

    Document
      ↓
    Structured document representation

That distinction is extremely important for your RAG journey.

### Step 12 — Compare Basic OCR vs Docling

Now let's explicitly compare the two approaches.

Approach A — Basic Tesseract
Scanned PDF
     ↓
Page Image
     ↓
Tesseract
     ↓
Plain Text

Output:

"Employee Details Name Department..."
Approach B — Docling
Scanned PDF
     ↓
Docling
     ↓
OCR + document processing
     ↓
DoclingDocument
     ↓
Markdown / JSON / other representations

Potentially:

# Employee Details


| Name | Department | Experience |
|---|---|---|
| Ravi | Data | 8 |
| Suresh | IT | 10 |

The second representation is much more useful for downstream RAG processing.

Docling's PDF pipeline also supports table-structure processing, which is one reason it is more appropriate for complex documents than treating everything as plain OCR text.

### Step 13 — Check Whether OCR Actually Happened

This is an important experiment.

Our input is a scanned PDF.

Therefore, we want to see whether Docling was able to recover text from the page.

Start with:

In [ ]:
print(markdown_text[:3000])

Look at the output.

If you see meaningful text from the scanned document, Docling successfully processed the scanned content.

### Step 14 — Understand What Docling Is Doing for Us

This is the key learning point.

Previously:

    PDF
    ↓
    render page
    ↓
    PNG
    ↓
    Tesseract
    ↓
    text

Now:

    PDF
    ↓
    Docling PDF pipeline
    ↓
    document processing
    ├── OCR
    ├── layout
    ├── reading order
    ├── table structure
    └── document representation
    ↓
    DoclingDocument

That's why I recommended moving toward Docling.

### Step 15 — But Don't Assume Docling Solves Everything

This is important for your production-grade RAG learning.

Docling is not magic.

For example:

Scanned architecture diagram

is different from:

Scanned paragraph

And:

Mathematical formula

is different from:

Table

Docling has additional enrichment capabilities and different processing pipelines for more advanced scenarios. Its current CLI, for example, exposes options for formula enrichment, picture description, chart extraction and other processing capabilities.

We'll learn those after we understand the basic Docling pipeline.

### Step 16 — Our First RAG-Oriented Architecture

At this point, our architecture becomes:

                    DOCUMENT
                       │
                       ▼
                    Docling
                       │
             ┌─────────┴─────────┐
             │                   │
          Text/Table          Images/etc.
             │
             ▼
       DoclingDocument
             │
             ▼
       Markdown / JSON
             │
             ▼
       RAG preprocessing
             │
             ▼
          Chunking
             │
             ▼
         Embeddings
             │
             ▼
        Vector Store

This is much closer to the architecture we'll eventually use in your production-grade RAG.

### Step 17 — What We Are NOT Doing Yet

For this stage, don't add:

    custom OCR preprocessing
    OpenCV
    image thresholding
    manual Tesseract configuration
    table extraction tuning
    formula extraction
    chart extraction
    VLM pipelines
    LangChain integration
    chunking

Those belong to later stages.

We're learning one thing now:

How can Docling ingest a scanned PDF and turn it into a structured document representation?

### Step 18 — Your Notebook Cells for This Stage

You have created a different notebook for this step implementation

### Step 19 — One Important Thing About OCR Configuration

There's a subtle point here.

Docling's current PDF pipeline supports OCR, but its OCR configuration is customizable. The documentation exposes PdfPipelineOptions, including do_ocr and OCR engine options.

So eventually we can explicitly configure:

      Docling
         │
         ▼
      PDF Pipeline
         │
         ├── OCR enabled
         │
         ├── OCR engine = RapidOCR
         │
         ├── OCR engine = Tesseract
         │
         └── OCR engine = EasyOCR

For this first experiment, however, I want you to see what the standard Docling pipeline gives us before we start changing its internals.

### Step 20 — What You've Achieved

This is the important transition in your RAG journey:

Before

You were learning:

How does OCR work?
Now

You're learning:

How do modern document-processing frameworks
handle OCR and complex documents for RAG?

And that's much closer to your actual goal.

Your current journey is now:

    Stage 1.4.2
    OCR & Complex PDF Ingestion
            │
            └── Stage 1.4.2.5
                OCR Approach
                    │
                    ├── 1.4.2.5.1
                    │   Basic OCR
                    │
                    └── 1.4.2.5.2
                        Docling for Scanned PDFs ← CURRENT

Run Steps 1–8 first. In particular, after result = converter.convert(pdf_path), inspect the Markdown output from your actual scanned PDF. That output will determine what we investigate next rather than blindly moving ahead.